# BeamZ 3D Viewer Demo

This notebook builds a small 3D BeamZ design and renders it inline with the integrated `beamz.visual.scene` viewer.

In [1]:
from pathlib import Path
import sys

repo_root = Path.cwd()
if not (repo_root / "beamz").exists() and (repo_root.parent / "beamz").exists():
    repo_root = repo_root.parent
sys.path.insert(0, str(repo_root))

try:
    import anywidget  # noqa: F401
except ImportError as exc:
    raise RuntimeError(
        "Inline notebook rendering requires anywidget. Reinstall BeamZ or install anywidget in this environment."
    ) from exc

import numpy as np

from IPython.display import display
from beamz import Design, GaussianSource, Material, Monitor, Rectangle, um, µm, LIGHT_SPEED, Taper
from beamz.visual.scene import view3d


In [ ]:
# Match 2D geometry in x/y; extend to a realistic 3D stack in z:
# substrate + cladding + air with enough buffer to keep core away from PML.
X, Y = 20 * µm, 10 * µm
Z_SUBSTRATE = 1.5 * µm
Z_CLADDING = 1.0 * µm
Z_AIR = 1.5 * µm
Z = Z_SUBSTRATE + Z_CLADDING + Z_AIR
WL = 1.55 * µm
TIME = 40 * WL / LIGHT_SPEED
N_CORE, N_CLAD, N_AIR = 2.04, 1.444, 1.0
WG_W = 0.565 * µm
H, W, OFFSET = 3.5 * µm, 9 * µm, 1.05 * µm
MMI_TAPER = 1.5 * µm
# Silicon-like core thickness inside the cladding region.
WG_T = 0.22 * µm
WG_Z0 = Z_SUBSTRATE
PML_THICKNESS = 1.2 * WL
PML_EDGES = "all"

clad = Material(1.44**2)
si = Material(3.47**2)
nitride = Material(2.0**2)

# Build 3D design with explicit material stack in z.
design = Design(width=X, height=Y, depth=Z, material=Material(N_AIR**2))
z0 = WG_Z0
mmi_body_start = X / 2 - W / 2 + MMI_TAPER

# Bottom substrate
design += Rectangle(
    position=(0, 0, 0),
    width=X,
    height=Y,
    depth=Z_SUBSTRATE,
    material=Material(N_CLAD**2),
)
# Middle cladding slab (core sits inside this region)
#design += Rectangle(
#    position=(0, 0, Z_SUBSTRATE),
#    width=X,
#    height=Y,
#    depth=Z_CLADDING,
#    material=Material(N_CLAD**2),
#)

design += Rectangle(
    position=(0, Y / 2 - WG_W / 2, z0),
    width=mmi_body_start,
    height=WG_W,
    depth=WG_T,
    material=Material(N_CORE**2),
)
design += Taper(
    position=(mmi_body_start, Y / 2, z0),
    input_width=WG_W,
    output_width=H,
    length=MMI_TAPER,
    depth=WG_T,
    material=Material(N_CORE**2),
)
design += Rectangle(
    position=(mmi_body_start + MMI_TAPER, Y / 2 - H / 2, z0),
    width=W - MMI_TAPER,
    height=H,
    depth=WG_T,
    material=Material(N_CORE**2),
)
design += Rectangle(
    position=(X / 2, Y / 2 + OFFSET - WG_W / 2, z0),
    width=X / 2,
    height=WG_W,
    depth=WG_T,
    material=Material(N_CORE**2),
)
design += Rectangle(
    position=(X / 2, Y / 2 - OFFSET - WG_W / 2, z0),
    width=X / 2,
    height=WG_W,
    depth=WG_T,
    material=Material(N_CORE**2),
)

view3d(design)

If you want a standalone browser page instead, use:

```python
view3d(design, mode="browser")
```